# AeroRUL — Phase 1: C-MAPSS Dataset Understanding

**Objective**: Understand, validate, organize, and document the NASA C-MAPSS dataset for subsequent RUL prediction modeling.

**Dataset**: NASA C-MAPSS (Commercial Modular Aero-Propulsion System Simulation)

**Reference**: Saxena et al., *Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation*, PHM08

---

## 1. Project Context

The goal of the AeroRUL project is to estimate the **Remaining Useful Life (RUL)** of turbofan aircraft engines from multivariate sensor degradation patterns.

The C-MAPSS dataset contains simulated run-to-failure data for turbofan engines under varying operating conditions and fault modes:

| Subset | Operating Conditions | Fault Modes |
|--------|---------------------|-------------|
| FD001  | 1 (Sea Level)       | 1 (HPC)     |
| FD002  | 6                   | 1 (HPC)     |
| FD003  | 1 (Sea Level)       | 2 (HPC+Fan) |
| FD004  | 6                   | 2 (HPC+Fan) |

**Phase 1** focuses exclusively on data understanding — no modeling is performed.

## 2. Imports

In [3]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    AVAILABLE_SUBSETS, COLUMN_NAMES, SENSOR_COLUMNS,
    OP_SETTING_COLUMNS, SUBSET_METADATA, FIGURES_DIR
)
from src.data_loader import discover_datasets, load_cmapss, load_all_subsets
from src.data_validation import (
    validate_structure, validate_missing_values, validate_engines,
    get_engine_final_cycles, validate_sensors, analyze_operating_conditions,
    sensor_cycle_correlation, sensor_correlation_matrix, trajectory_summary,
    run_full_validation
)
from src.visualization import (
    plot_trajectory_lengths, plot_sensor_distributions,
    plot_sensor_correlation_heatmap, plot_sensor_trajectories,
    plot_operating_conditions, plot_degradation_examples,
    plot_sensor_cycle_correlation
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)
%matplotlib inline

print(f"Project root: {PROJECT_ROOT}")

ModuleNotFoundError: No module named 'src'

## 3. Dataset Discovery

In [ ]:
# Discover which C-MAPSS subsets are available on disk
discovery = discover_datasets()

for subset, avail in discovery.items():
    meta = SUBSET_METADATA.get(subset, {})
    print(f"{subset}: train={avail['train']}, test={avail['test']}, rul={avail['rul']}")
    print(f"  -> {meta.get('description', 'N/A')}")

## 4. Loading C-MAPSS

In [ ]:
# Load all available subsets
all_data = load_all_subsets()

for subset, data in all_data.items():
    print(f"{subset}: train={data['train'].shape}, test={data['test'].shape}, rul={'available' if data['rul'] is not None else 'missing'}")

# Quick look at FD001 training data
print("\n--- FD001 Train (first 5 rows) ---")
all_data['FD001']['train'].head()

## 5. Schema Validation

In [ ]:
# Validate structure for all subsets
for subset, data in all_data.items():
    for split_name, df in [('train', data['train']), ('test', data['test'])]:
        result = validate_structure(df, f"{subset}_{split_name}")
        status = 'PASS' if result['column_count_ok'] and result['column_names_ok'] else 'FAIL'
        print(f"{subset} {split_name}: {result['n_rows']} rows x {result['n_cols']} cols | "
              f"Schema: {status} | Dup rows: {result['n_duplicate_rows']} | "
              f"Dup (unit,cycle): {result['n_duplicate_unit_cycle']}")

## 6. Data Quality Checks

In [ ]:
# Missing value check for all subsets
for subset, data in all_data.items():
    for split_name, df in [('train', data['train']), ('test', data['test'])]:
        missing = validate_missing_values(df)
        total_missing = missing['n_total_missing'].sum()
        print(f"{subset} {split_name}: {total_missing} total missing/infinite values")
        if total_missing > 0:
            print(missing[missing['n_total_missing'] > 0])

## 7. Training Trajectory Analysis

In [ ]:
# Training trajectory statistics
train_summaries = []
for subset, data in all_data.items():
    ts = trajectory_summary(data['train'], f"{subset}_train")
    train_summaries.append(ts)
    engines = validate_engines(data['train'])
    
    print(f"\n{'='*50}")
    print(f"{subset} TRAINING TRAJECTORIES")
    print(f"{'='*50}")
    print(f"  Engines:          {ts['n_engines']}")
    print(f"  Total obs:        {ts['n_observations']}")
    print(f"  Avg length:       {ts['avg_trajectory_length']}")
    print(f"  Min length:       {ts['min_trajectory_length']}")
    print(f"  Max length:       {ts['max_trajectory_length']}")
    print(f"  Median length:    {ts['median_trajectory_length']}")
    print(f"  Cycles ordered:   {engines['cycles_ordered']}")
    print(f"  Cycles start @1:  {engines['cycles_start_at_1']}")

In [ ]:
# Engine final cycles (FD001 example)
final_cycles = get_engine_final_cycles(all_data['FD001']['train'])
print("FD001 — Engine Final Cycles (first 20 engines):")
final_cycles.head(20)

## 8. Test Trajectory Analysis

In [ ]:
# Test trajectory statistics
for subset, data in all_data.items():
    ts = trajectory_summary(data['test'], f"{subset}_test")
    engines = validate_engines(data['test'])
    
    print(f"\n{'='*50}")
    print(f"{subset} TEST TRAJECTORIES")
    print(f"{'='*50}")
    print(f"  Engines:          {ts['n_engines']}")
    print(f"  Total obs:        {ts['n_observations']}")
    print(f"  Avg length:       {ts['avg_trajectory_length']}")
    print(f"  Min length:       {ts['min_trajectory_length']}")
    print(f"  Max length:       {ts['max_trajectory_length']}")
    print(f"  Median length:    {ts['median_trajectory_length']}")
    
    # Inspect RUL file structure
    if data['rul'] is not None:
        rul = data['rul']
        print(f"  RUL entries:      {len(rul)}")
        print(f"  RUL range:        [{rul['rul'].min()}, {rul['rul'].max()}]")
        print(f"  RUL mean:         {rul['rul'].mean():.1f}")

print("\n--- Key distinction ---")
print("Training: engine observed until simulated failure (last cycle = failure)")
print("Test:     engine trajectory stops BEFORE failure (RUL > 0)")

## 9. Operating Condition Analysis

In [ ]:
# Operating condition analysis per subset
for subset, data in all_data.items():
    oc = analyze_operating_conditions(data['train'])
    print(f"\n{subset} Operating Conditions (Training):")
    print(oc.to_string(index=False))

print("\n--- Interpretation ---")
print("FD001/FD003: Single operating condition -> op_settings nearly constant")
print("FD002/FD004: Six operating conditions -> significant variation in op_settings")
print("This makes FD002/FD004 harder due to operating-regime confounding.")

## 10. Sensor Analysis

In [ ]:
# Sensor statistics for FD001
fd001_sensors = validate_sensors(all_data['FD001']['train'])
print("FD001 — Sensor Statistics (Training):")
fd001_sensors

In [ ]:
# Identify constant/near-constant sensors
for subset, data in all_data.items():
    sensors = validate_sensors(data['train'])
    constant = sensors[sensors['is_constant']]['sensor'].tolist()
    near_const = sensors[sensors['is_near_constant'] & ~sensors['is_constant']]['sensor'].tolist()
    print(f"{subset}: constant={constant}, near-constant={near_const}")

In [ ]:
# Sensor-cycle correlation (FD001 training)
corr = sensor_cycle_correlation(all_data['FD001']['train'])
print("FD001 — Sensor-Cycle Pearson Correlation:")
print(corr.sort_values('pearson_r', ascending=False).to_string(index=False))
print("\nNote: This is an initial exploratory statistic only.")
print("Sensor selection will be addressed in a later phase.")

## 11. Visualizations

In [ ]:
# 1. Trajectory length histograms
plot_trajectory_lengths(all_data)
print("Saved: trajectory_lengths.png")

In [ ]:
# 2. Sensor distributions (FD001)
plot_sensor_distributions(all_data['FD001']['train'], 'FD001')
print("Saved: sensor_distributions_FD001.png")

In [ ]:
# 3. Sensor correlation heatmap (FD001)
corr_matrix = sensor_correlation_matrix(all_data['FD001']['train'])
plot_sensor_correlation_heatmap(corr_matrix, 'FD001')
print("Saved: sensor_correlation_FD001.png")

In [ ]:
# 4. Sensor trajectories (FD001)
plot_sensor_trajectories(all_data['FD001']['train'], 'FD001')
print("Saved: sensor_trajectories_FD001.png")

In [ ]:
# 5. Operating condition distributions
plot_operating_conditions(all_data)
print("Saved: operating_conditions.png")

In [ ]:
# 6. Degradation examples (FD001)
plot_degradation_examples(all_data['FD001']['train'], 'FD001')
print("Saved: degradation_examples_FD001.png")

In [ ]:
# 7. Sensor-cycle correlation bar chart (FD001)
corr_df = sensor_cycle_correlation(all_data['FD001']['train'])
plot_sensor_cycle_correlation(corr_df, 'FD001')
print("Saved: sensor_cycle_correlation_FD001.png")

## 12. Phase 1 Findings

### Key Observations

1. **All four C-MAPSS subsets (FD001-FD004) are available** with complete train/test/RUL files.

2. **Data quality is excellent**: No missing values, no duplicates, cycles are monotonically increasing and start at 1 for all engines across all subsets.

3. **Schema is consistent**: All files have exactly 26 columns (unit_id, cycle, 3 op_settings, 21 sensors).

4. **Operating conditions**:
   - FD001/FD003: Single condition (op_settings near-constant)
   - FD002/FD004: Six conditions (significant op_setting variation, confounding sensor readings)

5. **Constant/near-constant sensors**: Several sensors (e.g., sensor_1, sensor_5, sensor_10, sensor_16, sensor_18, sensor_19) appear constant or near-constant in FD001/FD003. These may need removal in later phases, but are preserved for now.

6. **Sensor degradation trends**: Many sensors show clear monotonic trends with cycle, indicating degradation signal.

### Recommended Next Phase

**Phase 2** should focus on:
- RUL target generation (remaining cycles for training data)
- RUL clipping experiments (e.g., max RUL = 125)
- Leakage-safe engine-level train/validation splitting
- Operating-condition normalization for FD002/FD004